In [1]:
# Test data repair scripts for NY

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))

from ny_data_repair import data_repair
MY_JURISDICTION = "New York"

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_top50_sample.parquet")


In [2]:
df = pd.read_parquet(INPUT_FILEPATH)
sub_df = df[df["JURISDICTION"] == MY_JURISDICTION]

for col in ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    sub_df[f'{col}_FLAG'] = ""

#sub_df_filled = sub_df.copy()
sub_df_filled = data_repair(sub_df)

assert(len(sub_df) == len(sub_df_filled))


In [3]:
print(f"FILE_DATE available (all): {sub_df['FILE_DATE'].notna().mean():.1%} -> {sub_df_filled['FILE_DATE'].notna().mean():.1%}")

print(f"PERMIT_DATE available (all): {sub_df['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled['PERMIT_DATE'].notna().mean():.1%}")

print(f"FINAL_DATE available (all): {sub_df['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled['FINAL_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Active', 'Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Active', 'Final'])
print(f"PERMIT_DATE available (active/final): {sub_df.loc[mask1]['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['PERMIT_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Final'])
print(f"FINAL_DATE available (final): {sub_df.loc[mask1]['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['FINAL_DATE'].notna().mean():.1%}")


FILE_DATE available (all): 87.1% -> 99.9%
PERMIT_DATE available (all): 63.1% -> 90.2%
FINAL_DATE available (all): 0.0% -> 43.9%
PERMIT_DATE available (active/final): 77.5% -> 97.8%
FINAL_DATE available (final): 0.0% -> 99.0%


In [4]:
print(sub_df['STATUS_NORMALIZED'].value_counts())
print(sub_df_filled['STATUS_NORMALIZED'].value_counts())

STATUS_NORMALIZED
Final        882
Active       569
In Review     86
Inactive      56
Name: count, dtype: int64
STATUS_NORMALIZED
Active       950
Final        887
In Review    104
Inactive      59
Name: count, dtype: int64


In [5]:
mask = sub_df_filled["PERMIT_DATE"].isna() 
#mask = sub_df_filled["JURISDICTION"].notna()
sample = sub_df_filled.loc[mask].sample(1).iloc[0]
DATA = sample["DATA"]
DATES_DATA = du.extract_date_fields(DATA) 

print(f"STATUS_NORMALIZED: {sample['STATUS_NORMALIZED']}    *Filled: {sample['STATUS_NORMALIZED_FLAG']}*")
print(f"RECORD_TYPE_ORIGINAL: {sample['RECORD_TYPE_ORIGINAL']}")
print(f"FILE_DATE: {sample['FILE_DATE']}       *Filled: {sample['FILE_DATE_FLAG']}*")
print(f"PERMIT_DATE: {sample['PERMIT_DATE']}   *Filled: {sample['PERMIT_DATE_FLAG']}*")
print(f"FINAL_DATE: {sample['FINAL_DATE']}     *Filled: {sample['FINAL_DATE_FLAG']}*")

print("DATES_DATA: ")
print(json.dumps(DATES_DATA, indent=2))



STATUS_NORMALIZED: In Review    *Filled: FILLED*
RECORD_TYPE_ORIGINAL: nan
FILE_DATE: 2020-10-15       *Filled: nan*
PERMIT_DATE: None   *Filled: nan*
FINAL_DATE: None     *Filled: nan*
DATES_DATA: 
{
  "filing": {
    "filing_date": "2020-10-15T00:00:00.000",
    "filing_status": "Objections",
    "current_status_date": "2020-12-27T20:49:42.000"
  }
}


In [6]:
print("DATA:")
print(json.dumps(json.loads(DATA), indent=2))



DATA:
{
  "filing": {
    "bin": "1078032",
    "lot": "1",
    "zip": "07058",
    "city": "PINE BROOK",
    "shed": "False",
    "sign": "False",
    "block": "323",
    "fence": "False",
    "state": "NJ",
    "antenna": "False",
    "borough": "MANHATTAN",
    "curb_cut": "False",
    "house_no": "72",
    "job_type": "Alteration",
    "little_e": "No",
    "scaffold": "False",
    "standpipe": "False",
    "filing_date": "2020-10-15T00:00:00.000",
    "street_name": "BARUCH DRIVE",
    "initial_cost": "384927",
    "building_type": "Other",
    "filing_status": "Objections",
    "work_on_floor": "Cellar, Cellar, Floor Number(s) 1 through 13",
    "commmunity_board": "103",
    "applicant_license": "055232",
    "exempt_from_nycecc": "True",
    "plumbing_work_type": "False",
    "applicant_last_name": "Shenoy",
    "current_status_date": "2020-12-27T20:49:42.000",
    "owner_s_street_name": "39, US HWY 46E",
    "sprinkler_work_type": "False",
    "applicant_first_name": "Ravi",
 